# Cosmos Proxy (Kava Equilibre) Second-Level Research

This notebook runs the dedicated `cosmos_second_level.py` pipeline over the **last 3 days** at **1-second resolution**. The charts below are arranged to answer three questions quickly: **is there an opportunity, at what size, and during which hours/windows?**

Important: this remains a **Cosmos proxy via Kava EVM**, not a direct Osmosis backtest.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from cosmos_second_level import (
    CHAIN_NAME,
    MARKET_KEY,
    PAIR_LABEL,
    PRIMARY_TRADE_SIZE_QUOTE,
    SAMPLE_WINDOW_DAYS,
    STATE_FREQUENCY,
    TRADE_SIZES_QUOTE,
    load_or_build_cosmos_dataset,
)

plt.style.use('default')

REFRESH_COSMOS_DATA = False
market_dataset = load_or_build_cosmos_dataset(PROJECT_ROOT, refresh=REFRESH_COSMOS_DATA, verbose=True)
manifest = market_dataset['dataset_manifest']
size_sensitivity = market_dataset['size_sensitivity'].copy().sort_values('trade_size_quote').reset_index(drop=True)
arb_labels = market_dataset['arb_labels'].copy().sort_values('timestamp').reset_index(drop=True)
opportunity_windows = market_dataset['opportunity_windows'].copy()

summary_rows = [
    {'field': 'chain_name', 'value': CHAIN_NAME},
    {'field': 'market_key', 'value': MARKET_KEY},
    {'field': 'pair_label', 'value': PAIR_LABEL},
    {'field': 'sample_window_days', 'value': SAMPLE_WINDOW_DAYS},
    {'field': 'state_frequency', 'value': STATE_FREQUENCY},
    {'field': 'window_start_utc', 'value': manifest['window_start_utc']},
    {'field': 'window_end_utc', 'value': manifest['window_end_utc']},
    {'field': 'primary_trade_size_quote', 'value': PRIMARY_TRADE_SIZE_QUOTE},
    {'field': 'trade_sizes_quote', 'value': ', '.join(str(x) for x in TRADE_SIZES_QUOTE)},
    {'field': 'gas_units', 'value': manifest['gas_units']},
    {'field': 'gas_estimation_source', 'value': manifest['gas_estimation']['gas_estimation_source']},
    {'field': 'positive_seconds', 'value': manifest['positive_seconds']},
    {'field': 'positive_windows', 'value': manifest['positive_windows']},
]

primary_row = size_sensitivity.loc[np.isclose(size_sensitivity['trade_size_quote'], PRIMARY_TRADE_SIZE_QUOTE)].iloc[0]
largest_positive_size = size_sensitivity.loc[size_sensitivity['net_positive_seconds'] > 0, 'trade_size_quote'].max() if (size_sensitivity['net_positive_seconds'] > 0).any() else np.nan
primary_positive_rate = (primary_row['net_positive_seconds'] / primary_row['non_stale_seconds'] * 100.0) if primary_row['non_stale_seconds'] else 0.0
if primary_row['net_positive_seconds'] <= 0:
    verdict = 'No net-positive route found at the primary size.'
elif largest_positive_size <= PRIMARY_TRADE_SIZE_QUOTE:
    verdict = 'Yes, but only at micro size; the edge disappears quickly as size increases.'
else:
    verdict = 'Yes, and the route survives above the primary size.'

verdict_rows = [
    {'field': 'verdict', 'value': verdict},
    {'field': 'primary_size_non_stale_seconds', 'value': int(primary_row['non_stale_seconds'])},
    {'field': 'primary_size_net_positive_seconds', 'value': int(primary_row['net_positive_seconds'])},
    {'field': 'primary_size_positive_rate_pct', 'value': round(primary_positive_rate, 3)},
    {'field': 'largest_trade_size_with_positive_seconds', 'value': largest_positive_size},
]

display(pd.DataFrame(summary_rows))
display(Markdown('## Opportunity Verdict'))
display(pd.DataFrame(verdict_rows))
display(Markdown('## Selected Pools'))
display(market_dataset['selected_pairs'])
display(Markdown('## Inferred Pool Fees'))
display(market_dataset['fee_summary'])
display(Markdown('## QC Report'))
display(pd.DataFrame(market_dataset['qc_report'].items(), columns=['check', 'value']))


In [ ]:
display(Markdown('## Opportunity Tables'))

display(Markdown('### Size Sensitivity'))
display(size_sensitivity)

display(Markdown('### Top Seconds'))
top_seconds = arb_labels.sort_values('net_edge_bps', ascending=False).head(25)
display(top_seconds[[
    'timestamp', 'route_name', 'buy_dex', 'sell_dex', 'conversion_dex', 'trade_size_quote',
    'gross_edge_bps', 'fee_cost_bps', 'gas_cost_quote', 'net_edge_quote', 'net_edge_bps',
    'opportunity_flag', 'stale_state'
]])

display(Markdown('### Positive Windows'))
display(opportunity_windows.head(25))

display(Markdown('### Positive Counts By Route'))
route_counts = (
    arb_labels.groupby('route_name', as_index=False)
    .agg(
        observed_seconds=('timestamp', 'count'),
        non_stale_seconds=('stale_state', lambda s: int((~s).sum())),
        gross_positive_seconds=('gross_edge_quote', lambda s: int((s > 0).sum())),
        net_positive_seconds=('opportunity_flag', 'sum'),
        max_net_edge_bps=('net_edge_bps', 'max'),
        max_net_profit_quote=('net_edge_quote', 'max'),
    )
    .sort_values(['net_positive_seconds', 'max_net_edge_bps'], ascending=[False, False])
    .reset_index(drop=True)
)
route_counts['positive_rate_pct'] = np.where(route_counts['non_stale_seconds'] > 0, route_counts['net_positive_seconds'] / route_counts['non_stale_seconds'] * 100.0, np.nan)
display(route_counts)


In [ ]:
pool_state = market_dataset['pool_state'].copy().sort_values('timestamp').reset_index(drop=True)

display(Markdown('## Opportunity Dashboard'))

size_plot = size_sensitivity.copy()
size_plot['gross_positive_rate_pct'] = np.where(size_plot['non_stale_seconds'] > 0, size_plot['gross_positive_seconds'] / size_plot['non_stale_seconds'] * 100.0, np.nan)
size_plot['net_positive_rate_pct'] = np.where(size_plot['non_stale_seconds'] > 0, size_plot['net_positive_seconds'] / size_plot['non_stale_seconds'] * 100.0, np.nan)
size_plot['trade_size_label'] = size_plot['trade_size_quote'].map(lambda x: f'{x:g}')

hourly = (
    arb_labels.assign(
        hour=arb_labels['timestamp'].dt.floor('1h'),
        non_stale=(~arb_labels['stale_state']).astype(int),
        positive=arb_labels['opportunity_flag'].astype(int),
    )
    .groupby('hour', as_index=False)
    .agg(
        non_stale_seconds=('non_stale', 'sum'),
        positive_seconds=('positive', 'sum'),
        max_net_edge_bps=('net_edge_bps', 'max'),
        mean_net_edge_bps=('net_edge_bps', 'mean'),
    )
)
hourly['positive_rate_pct'] = np.where(hourly['non_stale_seconds'] > 0, hourly['positive_seconds'] / hourly['non_stale_seconds'] * 100.0, np.nan)

if not opportunity_windows.empty:
    best_window = opportunity_windows.iloc[0]
    focus_mask = (arb_labels['timestamp'] >= best_window['start_timestamp']) & (arb_labels['timestamp'] <= best_window['end_timestamp'])
    focus = arb_labels.loc[focus_mask].sort_values('timestamp')
    focus_title = f"Best positive window: {best_window['seconds']} seconds"
else:
    focus = arb_labels.nlargest(min(900, len(arb_labels)), 'net_edge_bps').sort_values('timestamp')
    focus_title = 'Top scored seconds by net edge'

fig, axes = plt.subplots(2, 2, figsize=(18, 11), constrained_layout=True)

axes[0, 0].plot(size_plot['trade_size_label'], size_plot['gross_positive_rate_pct'], marker='o', linewidth=2, label='Gross positive share')
axes[0, 0].plot(size_plot['trade_size_label'], size_plot['net_positive_rate_pct'], marker='o', linewidth=2, label='Net positive share')
axes[0, 0].set_title('Positive share by trade size')
axes[0, 0].set_ylabel('% of non-stale seconds')
axes[0, 0].set_xlabel('Trade size (quote token)')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

axes[0, 1].bar(size_plot['trade_size_label'], size_plot['max_net_edge_bps'], alpha=0.8, label='Max net edge (bps)')
axes[0, 1].plot(size_plot['trade_size_label'], size_plot['mean_net_edge_bps'], color='black', marker='o', linewidth=1.8, label='Mean net edge (bps)')
axes[0, 1].axhline(0.0, color='red', linestyle='--', linewidth=1)
axes[0, 1].set_title('Edge by trade size')
axes[0, 1].set_ylabel('Basis points')
axes[0, 1].set_xlabel('Trade size (quote token)')
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend()

axes[1, 0].plot(hourly['hour'], hourly['positive_seconds'], marker='o', linewidth=1.8, label='Positive seconds')
hourly_rate_ax = axes[1, 0].twinx()
hourly_rate_ax.plot(hourly['hour'], hourly['positive_rate_pct'], color='tab:orange', linewidth=1.8, label='Positive rate (%)')
axes[1, 0].set_title('Hourly clustering of positive seconds')
axes[1, 0].set_ylabel('Positive seconds')
hourly_rate_ax.set_ylabel('Positive rate (%)')
axes[1, 0].grid(alpha=0.3)
lines_a, labels_a = axes[1, 0].get_legend_handles_labels()
lines_b, labels_b = hourly_rate_ax.get_legend_handles_labels()
axes[1, 0].legend(lines_a + lines_b, labels_a + labels_b, loc='upper right')

axes[1, 1].plot(focus['timestamp'], focus['gross_edge_bps'], linewidth=1.4, label='Gross edge (bps)')
axes[1, 1].plot(focus['timestamp'], focus['net_edge_bps'], linewidth=1.8, label='Net edge (bps)')
axes[1, 1].fill_between(focus['timestamp'], 0, focus['net_edge_bps'], where=focus['opportunity_flag'].to_numpy(), alpha=0.25, color='tab:green', label='Net-positive seconds')
axes[1, 1].axhline(0.0, color='red', linestyle='--', linewidth=1)
axes[1, 1].set_title(focus_title)
axes[1, 1].set_ylabel('Basis points')
axes[1, 1].set_xlabel('Timestamp (UTC)')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

plt.show()

display(Markdown('## Price Context'))
price_plot = (
    pool_state.assign(plot_timestamp=pool_state['timestamp'].dt.floor('5min'))
    .groupby(['plot_timestamp', 'dex'], as_index=False)
    .agg(mid_price_quote_per_base=('mid_price_quote_per_base', 'last'))
)
fig, ax = plt.subplots(figsize=(16, 4), constrained_layout=True)
for dex, group in price_plot.groupby('dex'):
    ax.plot(group['plot_timestamp'], group['mid_price_quote_per_base'], label=dex, linewidth=1.5)
ax.set_title('5-minute sampled pool prices')
ax.set_ylabel('Quote per base')
ax.set_xlabel('Timestamp (UTC)')
ax.grid(alpha=0.3)
ax.legend()
plt.show()


In [ ]:
manifest_path = PROJECT_ROOT / 'outputs' / 'cosmos_research' / 'metadata' / 'dataset_manifest.json'
display(Markdown(f'## Manifest\n\nSaved manifest: `{manifest_path}`'))
if manifest_path.exists():
    manifest_payload = json.loads(manifest_path.read_text())
    display(pd.json_normalize(manifest_payload, sep='.', max_level=1).T.reset_index().rename(columns={'index': 'field', 0: 'value'}))
